In [1]:
import os

from dotenv import load_dotenv

load_dotenv()  # Returns a boolean indicating whether the .env file was found and loaded successfully
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Add project root to sys.path so `from src...` imports work,
# regardless of where Jupyter's working directory happens to be
project_root = Path.cwd().parent  # assumes notebook runs from notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# invoked llm to test if the API key is working
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c6b5b9a933', 'id': 'chatcmpl-E7tjjrdxk0UbDXByyx4ASWrcVr8IV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fbb25-f17f-7bb0-b66c-2ed4e2f804c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from src.rag.retrieve import retrieve_relevant_documents
from src.data.state import AgriChainState

# Build a minimal test state — just enough for this function to read/write
test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}

result = retrieve_relevant_documents(test_state)  # type: ignore

# Inspect what came back
for doc in result["retrieved_documents"]: # type: ignore
    print(doc["score"], doc["doc_type"], doc["content"][:80])

c:\Users\mjhog\MyCode\fullstack-academy\agrichain_project\src\rag\retrieve.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1.1126602 supplier_info Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100.
1.1330247 supplier_info Supplier SUP-5814 - Europe Region: Specializes in tomatoes. Quality score: 76/10
1.1471378 supplier_info Supplier SUP-12905 - Europe Region: Specializes in tomatoes. Quality score: 66/1
1.3364317 supplier_info Supplier SUP-6479 Profile: Certified organic producer in Europe. Capacity: 218 t
1.4174404 supplier_info Supplier SUP-6050 Profile: Certified organic producer in North America. Capacity


In [8]:
import json
from collections import Counter

with open("../data/raw/knowledge_base.json") as f:
    kb = json.load(f)

print(Counter(doc["doc_type"] for doc in kb))

Counter({'supplier_info': 200, 'sop': 50, 'resolution_guide': 24})


In [11]:
from src.models.classify import predict_category

# create a test complaint text
test_complaint_text = "the invoice charged me twice for the same order"

category = predict_category(test_complaint_text)  # type: ignore
print(f"Predicted category: {category}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted category: Pricing Error


In [15]:
from src.agents.analyzer import analyze_severity

# Minimal state — only what this function actually reads
test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
test_state_two = {
    "complaint_text": "metal shavings, someone could get hurt",
}
result = analyze_severity(test_state)
result_two = analyze_severity(test_state_two)
print("Severity:", result["severity"])
print("Reasoning:", result["severity_reasoning"])
print("Severity:", result_two["severity"])
print("Reasoning:", result_two["severity_reasoning"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Severity: high
Reasoning: The complaint involves perishable goods (tomatoes) that are bruised and moldy, indicating they may spoil quickly and are no longer suitable for sale or consumption. This not only affects the immediate financial impact due to potential loss of product but also raises safety concerns regarding food contamination. Additionally, the quality issue could significantly damage the customer relationship, especially if this is a recurring problem.
Severity: high
Reasoning: The presence of metal shavings poses a significant safety risk, as it could lead to injury if the goods are used or consumed. This raises immediate concerns for customer safety and potential liability for the company. While the financial impact may not be directly quantifiable, the risk of harm to customers and the potential for damage to the company's reputation and customer relationships is substantial, warranting a high sev